# The Ancient Secrets of Prediction

You go to the market and buy 3 apples; with parking, the trip costs 6. Next week
you buy 5 apples; the trip costs 8. **How much will a 10-apple trip cost?**

Write the story as an equation:

$$Expense = parking + price \cdot apples \qquad\text{i.e.}\qquad y = \beta_1 x + \beta_0$$

Trip 1 says $3\beta_1 + \beta_0 = 6$; trip 2 says $5\beta_1 + \beta_0 = 8$.
Subtract them: $2\beta_1 = 2$, so the price is 1 and parking is 3. Ten apples: **13**.

That is the oldest prediction machine there is: *turn observations into equations,
solve, extrapolate*. Long before gradient descent, this is how curves were fit —
and in this chapter you rebuild the whole machine, then push it to its breaking
point (data that no curve passes through) and invent **least squares**.

**Prerequisite:** the Loops and Arrays chapter — you will reuse your
`solve_equations` from its Part 5.

---
## Part 1 — Vectors: the Arithmetic of Prediction

A fitted model is just a list of coefficients — a **vector**. Evaluating it at a
point is a *dot product*. Build that toolkit first.

### Exercise 1.1 — `dot_product`

`dot_product(a, b)` multiplies element-by-element and sums:
`dot_product([1, 2], [3, 4])` is `1*3 + 2*4 = 11`.

**Write your solution in `exercises/ex_1_1_dot_product.py`**

In [ ]:
def dot_product(a, b):
    pass  # replace this


assert dot_product([1, 2], [3, 4]) == 11
assert dot_product([1, 0, 2], [5, 5, 5]) == 15
assert dot_product([2], [7]) == 14
print("dot_product: OK")

### Exercise 1.2 — `vector_length` and `convert_to_unit`

- The length of a vector is the distance formula in disguise:
  $\|v\| = \sqrt{v \cdot v}$ — reuse your `dot_product`.
- A **unit vector** keeps the direction but has length 1: divide every component
  by the length. `convert_to_unit([3, 4])` → `[0.6, 0.8]`.

**Write your solution in `exercises/ex_1_2_vector_length.py`**

In [ ]:
import math

def vector_length(v):
    pass  # replace this

def convert_to_unit(v):
    pass  # replace this


assert vector_length([3, 4]) == 5.0
assert convert_to_unit([3, 4]) == [0.6, 0.8]
assert abs(vector_length(convert_to_unit([2, 3, 6])) - 1.0) < 1e-9
print("vector_length / convert_to_unit: OK")

### Exercise 1.3 — `cosine_similarity`

How *aligned* are two vectors? Take the dot product of their **unit** versions.
The result is the cosine of the angle between them:

- same direction → `1.0`
- perpendicular → `0.0`
- opposite → `-1.0`

This one number is how modern systems compare documents, faces and embeddings —
you will meet it again the moment you touch LLMs.

**Write your solution in `exercises/ex_1_3_cosine_similarity.py`**

In [ ]:
def cosine_similarity(a, b):
    pass  # replace this


assert abs(cosine_similarity([1, 2], [2, 4]) - 1.0) < 1e-9    # same direction
assert abs(cosine_similarity([1, 0], [0, 1])) < 1e-9           # perpendicular
assert abs(cosine_similarity([1, 2], [-1, -2]) + 1.0) < 1e-9   # opposite
print("cosine_similarity: OK")

### Exercise 1.4 — `matrix_mul`

A matrix is a list of rows. In the product `C = A @ B`, entry `C[i][j]` is the
dot product of **row i of A** with **column j of B**. An (n×k) matrix times a
(k×m) matrix gives an (n×m) matrix.

Write `matrix_mul(A, B)` using your `dot_product`.
(**Hint:** column `j` of `B` is `[row[j] for row in B]`.)

**Write your solution in `exercises/ex_1_4_matrix_mul.py`**

In [ ]:
def matrix_mul(A, B):
    pass  # replace this


A = [[1, 2],
     [3, 4]]
B = [[5, 6],
     [7, 8]]
assert matrix_mul(A, B) == [[19, 22], [43, 50]]

M = [[1, 2, 3],
     [4, 5, 6]]
v = [[1], [0], [2]]                      # a column vector
assert matrix_mul(M, v) == [[7], [16]]   # (2x3) @ (3x1) -> (2x1)
print("matrix_mul: OK")

---
## Part 2 — Bring Your Solver

We store the equation $a_1 x_1 + a_2 x_2 + \dots + a_n x_n = b$ as the list
`[a1, a2, ..., an, b]`, and a system as a list of such lists — exactly the
representation you used in **Loops and Arrays, Part 5**.

Paste your `solve_equations` (with its helpers `solve_for_first_variable`,
`eliminate_variable_pair`, `eliminate_variable`) into the next cell. The test
must pass before you go on. (Skipped that chapter? Go invent it there first —
this whole chapter stands on it.)

In [ ]:
# Paste your solver from the Loops and Arrays chapter here:
#   solve_for_first_variable, eliminate_variable_pair,
#   eliminate_variable, solve_equations


r = solve_equations([[1, 1, 1, 35], [3, 2, 1, 75], [4, 3, 1, 105]])
assert all(abs(a - b) < 1e-9 for a, b in zip(r, [10.0, 20.0, 5.0])), r
print("solver ready:", r)

**Think:** your solver eliminates one variable at a time, and each elimination
touches every remaining equation, and each equation has that many coefficients —
three nested loops. Roughly how does the running time grow with `n` equations?
If solving 4 equations takes 1 second, what would 8 take? Keep that number in
mind — it returns at the end of the chapter.

---
## Part 3 — Prediction as Equation Solving

### Exercise 3.1 — The Apple Trips, by Code

Redo the opening story with your tools:

1. Build the two equations in `(beta1, beta0)`: trip 1 → `[3, 1, 6]`,
   trip 2 → `[5, 1, 8]`.
2. `solve_equations` gives you `beta = [price, parking]`.
3. Predict the 10-apple trip with `dot_product(beta, [10, 1])`.

In [ ]:
# YOUR CODE — set up the equations, solve, predict the 10-apple trip
eqns = ...
beta = ...
prediction = ...

assert all(abs(a - b) < 1e-9 for a, b in zip(beta, [1.0, 3.0])), beta
assert abs(prediction - 13.0) < 1e-9
print("a 10-apple trip will cost:", prediction)

### Exercise 3.2 — When a Line Isn't Enough

Traffic through a junction is measured three times:

| t | traffic |
|---|---------|
| 1 | 6 |
| 2 | 12 |
| 3 | 20 |

The growth is not linear (check the gaps: +6, +8). Assume a quadratic:
$traffic = a t^2 + b t + c$. Each measurement is one equation in $(a, b, c)$ —
for $t=1$: $a + b + c = 6$, and so on.

Build the three equations, solve for `(a, b, c)`, and predict the traffic at `t = 4`.

In [ ]:
# YOUR CODE — three equations in (a, b, c), solve, predict t = 4
eqns = ...
abc = ...
prediction = ...

assert all(abs(x - y) < 1e-9 for x, y in zip(abc, [1.0, 3.0, 2.0])), abc
assert abs(prediction - 30.0) < 1e-9
print("traffic at t=4:", prediction)

### Exercise 3.3 — `solve_poly`: the General Machine

Automate it. Write `solve_poly(x, y, xp)`:

- `x` and `y` are `n` observed points; assume a polynomial of degree `n - 1`.
- For each point build the row
  `[x_i**(n-1), ..., x_i**2, x_i, 1, y_i]` and solve for the coefficients.
- Return the polynomial's value at `xp` — a `dot_product` of the coefficients
  with the matching powers of `xp`.

**Think:** why do `n` points pin down exactly a degree-`(n-1)` polynomial —
what goes wrong with fewer coefficients, and what freedom is left with more?

**Write your solution in `exercises/ex_3_3_solve_poly.py`**

In [ ]:
def solve_poly(x, y, xp):
    pass  # replace this


assert abs(solve_poly([1, 2, 3], [6, 12, 20], 4) - 30.0) < 1e-6
# a line is just a degree-1 polynomial — the apple trips again:
assert abs(solve_poly([3, 5], [6, 8], 10) - 13.0) < 1e-6
# and it must reproduce the points it was built from:
assert abs(solve_poly([1, 2, 3], [6, 12, 20], 2) - 12.0) < 1e-6
print("solve_poly: OK")

---
## Part 4 — When No Curve Fits

A factory reports:

| input | output |
|-------|--------|
| 1 | 8 |
| 2 | 11 |
| 3 | 16 |
| 4 | ?? |

Try a line through the first two points: slope 3, intercept 5 — but then input 3
predicts 14, not 16. Through points 1 and 3: slope 4, intercept 4 — but input 2
predicts 12, not 11. **No line passes through all three points.** Real
measurements are noisy; perfect solving just broke.

New plan: accept an imperfect line, but pick the one that is *least wrong*.
Define the total (squared) error of a candidate line $y = mx + c$:

$$E(m, c) = \sum_i (m x_i + c - y_i)^2$$

### Exercise 4.1 — `total_error`

Write `total_error(m, c, data)` where `data` is a list of `(x, y)` pairs.

**Write your solution in `exercises/ex_4_1_total_error.py`**

In [ ]:
factory_data = [(1, 8), (2, 11), (3, 16)]

def total_error(m, c, data):
    pass  # replace this


assert total_error(3, 5, factory_data) == 4.0   # hits two points, misses the third by 2
assert total_error(0, 0, factory_data) == 8*8 + 11*11 + 16*16
print("total_error: OK")

### The Best Line, Without Guessing

At the minimum of $E$, nudging $m$ or $c$ changes nothing — both partial
derivatives are zero. Differentiate (each term by the chain rule) and set to zero:

$$\frac{\partial E}{\partial m} = \sum_i 2(m x_i + c - y_i)\,x_i = 0
\qquad
\frac{\partial E}{\partial c} = \sum_i 2(m x_i + c - y_i) = 0$$

Divide by 2 and regroup — two *linear* equations in $(m, c)$, the **normal equations**:

$$m\textstyle\sum x_i^2 + c\sum x_i = \sum x_i y_i
\qquad
m\textstyle\sum x_i + c\,n = \sum y_i$$

For the factory data: $\sum x^2 = 14$, $\sum x = 6$, $\sum xy = 8 + 22 + 48 = 78$,
$\sum y = 35$, $n = 3$:

$$14m + 6c = 78 \qquad 6m + 3c = 35$$

### Exercise 4.2 — Solve Them

Use `solve_equations` on those two equations. The test also nudges your answer in
every direction to confirm no neighbor does better — the definition of a minimum.

In [ ]:
# YOUR CODE — solve the two normal equations for m and c
m, c = ...

assert abs(m - 4.0) < 1e-9, m
assert abs(c - 11/3) < 1e-9, c

best = total_error(m, c, factory_data)
for dm, dc in [(0.01, 0), (-0.01, 0), (0, 0.01), (0, -0.01)]:
    assert total_error(m + dm, c + dc, factory_data) > best
print(f"best line: y = {m:.3f}x + {c:.3f}   (error {best:.4f})")

### Exercise 4.3 — `best_line`: Least Squares for Any Data

Wrap it up. Write `best_line(data)`:

1. From the `(x, y)` pairs compute $\sum x^2$, $\sum x$, $\sum xy$, $\sum y$ and $n$.
2. Build the two normal equations and `solve_equations`.
3. Return `(m, c)`.

On data that *is* perfectly linear it must recover the exact line — least squares
contains exact fitting as a special case.

**Write your solution in `exercises/ex_4_3_best_line.py`**

In [ ]:
def best_line(data):
    pass  # replace this


m, c = best_line([(0, 3), (1, 5), (2, 7)])        # perfectly linear: y = 2x + 3
assert abs(m - 2.0) < 1e-9 and abs(c - 3.0) < 1e-9

m, c = best_line(factory_data)
assert abs(m - 4.0) < 1e-9 and abs(c - 11/3) < 1e-9
print("factory prediction for input 4:", round(m * 4 + c, 3))
print("best_line: OK")

In [ ]:
# Provided — run this cell as-is to see your line.
import matplotlib.pyplot as plt

xs = [p[0] for p in factory_data]
ys = [p[1] for p in factory_data]
m, c = best_line(factory_data)
line_x = [0.5, 4.2]
line_y = [m * x + c for x in line_x]

plt.scatter(xs, ys, s=70, zorder=3, label="measurements")
plt.plot(line_x, line_y, "r-", label=f"y = {m:.2f}x + {c:.2f}")
plt.scatter([4], [m * 4 + c], marker="*", s=200, color="green", zorder=3,
            label="prediction for input 4")
plt.xlabel("input"); plt.ylabel("output")
plt.title("Least squares: the least-wrong line")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

---
# Bonus — Matrices That Move the World

So far matrices held *data* (trips, prices). But `matrix_mul` has a second
life: a matrix can be a **machine that moves points**. Every video game, every
3D animation, every graphics chip is built on this.

Keep points as rows, `[x, y]`. Multiplying a stack of points by one 2×2
matrix moves *all* of them at once with your existing `matrix_mul` — no new
code needed.

### Exercise R.1 — The rotation matrix

The matrix that rotates every point counter-clockwise by angle $\theta$
(with points as rows) is:

$$R(\theta) = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix}$$

Write `rotation_matrix(angle_degrees)` returning that 2×2 list-of-lists.
Use `math.radians` to convert, then `math.cos` / `math.sin`.

Sanity checks worth predicting before you run: the point `[1, 0]` (due east)
rotated 90° should land at `[0, 1]` (due north). Rotated 180°, at `[-1, 0]`.

In [ ]:
import math

def rotation_matrix(angle_degrees):
    pass  # replace this


p = matrix_mul([[1, 0]], rotation_matrix(90))[0]
assert abs(p[0]) < 1e-9 and abs(p[1] - 1) < 1e-9      # east -> north

p = matrix_mul([[1, 0]], rotation_matrix(180))[0]
assert abs(p[0] + 1) < 1e-9 and abs(p[1]) < 1e-9      # east -> west

p = matrix_mul([[0, 1]], rotation_matrix(90))[0]
assert abs(p[0] + 1) < 1e-9 and abs(p[1]) < 1e-9      # north -> west
print("rotation_matrix: OK")

### Exercise R.2 — Turn a whole shape

Here is a kite, five points as rows (the last repeats the first so it draws
closed):

```python
kite = [[0, 0], [1, 2], [0, 5], [-1, 2], [0, 0]]
```

Rotate the *entire* kite by 45° with **one** call to `matrix_mul`, storing the
result in `turned`. Then run the provided plotting cell to see both.

A deep property to verify while you're here: rotation moves points but never
stretches them. The test checks every point stays the same distance from the
origin.

In [ ]:
kite = [[0, 0], [1, 2], [0, 5], [-1, 2], [0, 0]]

turned = ...   # replace this — one matrix_mul call

# Tests: same number of points, all distances to origin preserved
assert len(turned) == len(kite)
for before, after in zip(kite, turned):
    d_before = (before[0] ** 2 + before[1] ** 2) ** 0.5
    d_after = (after[0] ** 2 + after[1] ** 2) ** 0.5
    assert abs(d_before - d_after) < 1e-9
print("rotation preserves all lengths — the shape is rigid")

In [ ]:
# Provided — run this cell as-is to see the kite and its rotation.
import matplotlib.pyplot as plt

plt.plot([p[0] for p in kite], [p[1] for p in kite], "b-o", label="original")
plt.plot([p[0] for p in turned], [p[1] for p in turned], "r-o", label="rotated 45°")
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.gca().set_aspect("equal")
plt.legend(); plt.grid(alpha=0.3)
plt.title("One matrix multiplication moves every point")
plt.show()

### Exercise R.3 — What does a *random* matrix do?

Rotation matrices are special. What happens with an arbitrary 2×2 matrix?

Build one from random numbers, apply it to the kite with `matrix_mul`, and
plot the result (reuse the plotting code above). Run it several times.

**Observe, then answer:** straight edges stay straight, and parallel edges
stay parallel — but lengths and angles change: the kite gets stretched,
squashed, sheared, maybe flipped. Every 2×2 matrix is some such *linear
transformation*, and rotations are exactly the ones that preserve all
lengths. (When a neural network multiplies data by a learned weight matrix,
this stretching and squashing of space is literally what it's doing.)

In [ ]:
import random

R = [[random.uniform(-1, 1), random.uniform(-1, 1)],
     [random.uniform(-1, 1), random.uniform(-1, 1)]]
warped = ...   # replace this — apply R to the kite

# Test: still 5 points, each with 2 coordinates
assert len(warped) == 5 and all(len(p) == 2 for p in warped)

plt.plot([p[0] for p in kite], [p[1] for p in kite], "b-o", label="original")
plt.plot([p[0] for p in warped], [p[1] for p in warped], "g-o", label="warped")
plt.gca().set_aspect("equal")
plt.legend(); plt.grid(alpha=0.3)
plt.title("A random matrix: straight lines stay straight, nothing else survives")
plt.show()

---
## Summary

| You built | The secret |
|-----------|------------|
| `dot_product`, `vector_length`, `convert_to_unit`, `cosine_similarity`, `matrix_mul` | models are vectors; evaluation is a dot product |
| apple trips, traffic at the junction | *prediction = set up equations + solve* |
| `solve_poly` | n points ↔ one degree-(n−1) polynomial |
| `total_error`, normal equations, `best_line` | **least squares** — when nothing fits, minimize the squared error |

What you just derived is **Ordinary Least Squares**, the same closed-form answer
`sklearn.linear_model.LinearRegression` computes.

So why does the next chapter exist? Two cracks in the ancient method: solving
grows like $n^3$ (your Part 2 estimate), and the derive-set-to-zero trick needs a
formula for the derivative — which complicated models don't offer. **Gradient
Descent** fixes both by *walking* toward the minimum instead of jumping to it.